# BERTopic Thesis Reanalysis with Local Llama 3.1 Labeling (No API)

**Project:** *Mapping Narrative Patterns in News Media Using BERTopic: A Bibliometric and Topic Modeling Analysis of Scopus Data*

This notebook reproduces the abstract-based BERTopic analysis using the same corpus, preprocessing logic, embedding/clustering pipeline, metrics, tables, and figures as the OpenAI-assisted version. The only methodological change is the final topic-labeling stage. Instead of requiring an OpenAI API key, this version uses a local LLM through Ollama:

- abstract-only topic modeling;
- embedding robustness check and UMAP/HDBSCAN grid;
- c-TF-IDF topic representation with real term scores;
- outlier review;
- representative-document evidence for topic labeling;
- optional local Llama 3.1-assisted topic labeling with `llama3.1:8b`;
- automatic export of CSV, PNG, PDF, HTML, and LaTeX-ready outputs.

The local LLM step is used only after BERTopic has produced topics. It does not create embeddings, does not form clusters, and does not assign documents to topics. Its purpose is to provide a no-API alternative for reviewers who want to reproduce the topic-labeling procedure without access to the author's OpenAI key.


## 0. Installation

Run this cell in Google Colab or a fresh local environment. If packages are already installed, this step can be skipped.


In [ ]:
# Uncomment if needed:
# !pip -q install bertopic sentence-transformers umap-learn hdbscan pandas numpy matplotlib seaborn scikit-learn gensim openpyxl plotly kaleido adjustText spacy gdown requests
# !python -m spacy download en_core_web_sm

# Optional local LLM setup in Colab/Linux. This downloads and runs Ollama locally.
# It can take several minutes and requires enough disk/RAM.
# !curl -fsSL https://ollama.com/install.sh | sh
# !ollama pull llama3.1:8b


## 1. Reproducibility and Project Structure


In [ ]:
from __future__ import annotations

import itertools
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import textwrap
import warnings
from collections import Counter
from datetime import datetime
from pathlib import Path

warnings.filterwarnings("ignore")

SEED = 42
CPU_THREADS = int(os.environ.get("BERTOPIC_CPU_THREADS", "4"))
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMBA_NUM_THREADS"] = "1"

random.seed(SEED)

import numpy as np
import pandas as pd

np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.set_num_threads(CPU_THREADS)
except Exception:
    torch = None

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

PROJECT_NAME = "BERTopic_Abstract_Hyperparameter_Rebuild"
PREPROCESSING_VERSION = "v3_representation_only_boilerplate"

if IN_COLAB:
    PROJECT_ROOT = Path("/content") / PROJECT_NAME
    USE_DRIVE = False
    if USE_DRIVE:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME
else:
    PROJECT_ROOT = Path.home() / "Desktop" / PROJECT_NAME

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs_llama31_local"

for subdir in [
    "figures", "tables", "reports", "models", "html", "grid",
    "outliers", "labeling_evidence", "latex", "environment"
]:
    (OUTPUT_DIR / subdir).mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "scopus_final_with_references.csv"
THESAURUS_PATH = DATA_DIR / "thesaurus_bibliometrix_elian.csv"

SCOPUS_DRIVE_ID = "1hMfO2-pL64oikEK5EDttGd4IAMjMfNH-"
THESAURUS_DRIVE_ID = "1cFGaGtpBdwSikrRzZSosHWFtbPdf5N1_"

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH, DATA_PATH.exists())
print("Thesaurus path:", THESAURUS_PATH, THESAURUS_PATH.exists())
print("Seed:", SEED)
print("Preprocessing version:", PREPROCESSING_VERSION)


## 2. Data Access

The recommended structure is:

```text
BERTopic_Abstract_Hyperparameter_Rebuild/
|-- data/
|   |-- scopus_final_with_references.csv
|   `-- thesaurus_bibliometrix_elian.csv
|-- notebook/
`-- outputs/
```

If the files are not already present, the notebook first searches common local locations and then attempts to download them from the Google Drive file IDs provided for this project.


In [ ]:
def download_from_google_drive(file_id: str, destination: Path) -> None:
    import gdown
    destination.parent.mkdir(parents=True, exist_ok=True)
    url = f"https://drive.google.com/uc?id={file_id}"
    print(f"Downloading {destination.name} from Google Drive...")
    gdown.download(url, str(destination), quiet=False)


def copy_first_existing(candidates: list[Path], destination: Path) -> bool:
    for candidate in candidates:
        if candidate.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(candidate, destination)
            print(f"Copied {candidate} -> {destination}")
            return True
    return False


if not DATA_PATH.exists():
    local_candidates = [
        Path.home() / "Downloads" / "scopus_final_with_references (1).csv",
        Path.home() / "Downloads" / "scopus_final_with_references.csv",
        Path.home() / "Desktop" / "BERTopic_Hyperparameter_Grid_Review" / "data" / "scopus_final_with_references.csv",
        Path.home() / "Desktop" / "BERTopic_Thesis_Reanalysis_OpenAI" / "data" / "scopus_final_with_references.csv",
    ]
    copied = copy_first_existing(local_candidates, DATA_PATH)
    if not copied:
        download_from_google_drive(SCOPUS_DRIVE_ID, DATA_PATH)

if not THESAURUS_PATH.exists():
    local_candidates = [
        Path.home() / "Downloads" / "thesaurus_bibliometrix_elian (1).csv",
        Path.home() / "Downloads" / "thesaurus_bibliometrix_elian.csv",
        Path.home() / "Desktop" / "BERTopic_Hyperparameter_Grid_Review" / "data" / "thesaurus_bibliometrix_elian.csv",
        Path.home() / "Desktop" / "BERTopic_Thesis_Reanalysis_OpenAI" / "data" / "thesaurus_bibliometrix_elian.csv",
    ]
    copied = copy_first_existing(local_candidates, THESAURUS_PATH)
    if not copied:
        download_from_google_drive(THESAURUS_DRIVE_ID, THESAURUS_PATH)

print("Data path:", DATA_PATH, DATA_PATH.exists())
print("Thesaurus path:", THESAURUS_PATH, THESAURUS_PATH.exists())


## 3. Imports, Visual Style, and Helper Functions


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import patheffects as pe
from matplotlib.lines import Line2D
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import minmax_scale

from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer

try:
    from IPython.display import display, Image as IPImage
except Exception:
    IPImage = None
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)

try:
    from adjustText import adjust_text
except Exception:
    adjust_text = None

def running_in_notebook() -> bool:
    try:
        shell_name = get_ipython().__class__.__name__
        return IN_COLAB or shell_name in {"ZMQInteractiveShell", "Shell"}
    except Exception:
        return False

def show_figure(fig):
    if running_in_notebook():
        plt.show()
    plt.close(fig)

COLORS = {
    "teal": "#6FA8A8",
    "teal_dark": "#3F7F82",
    "teal_light": "#B7D5D5",
    "brown": "#8A6B4E",
    "brown_dark": "#5F4633",
    "gray_strip": "#D9D9D9",
    "grid": "#EAEAEA",
    "text": "#222222",
    "muted": "#666666",
    "red": "#C44E52",
    "green": "#55A868",
    "blue": "#4C72B0",
    "purple": "#8172B3",
    "gold": "#CCB44C",
}

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif", "Times New Roman", "Times"],
    "axes.edgecolor": "#A0A0A0",
    "axes.linewidth": 0.8,
    "axes.labelcolor": COLORS["text"],
    "xtick.color": COLORS["muted"],
    "ytick.color": COLORS["muted"],
    "figure.dpi": 140,
    "savefig.dpi": 320,
    "savefig.bbox": "tight",
})

sns.set_theme(style="whitegrid", rc={
    "font.family": "serif",
    "axes.edgecolor": "#A0A0A0",
    "grid.color": COLORS["grid"],
})

def clean_whitespace(value):
    if pd.isna(value):
        return ""
    value = str(value).replace("\u00a0", " ")
    value = re.sub(r"\s+", " ", value).strip()
    return value

def read_csv_robust(path: Path) -> pd.DataFrame:
    for encoding in ["utf-8-sig", "utf-8", "latin1"]:
        try:
            return pd.read_csv(path, encoding=encoding, low_memory=False)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, low_memory=False)

def find_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    lower_map = {col.lower().strip(): col for col in df.columns}
    for candidate in candidates:
        key = candidate.lower().strip()
        if key in lower_map:
            return lower_map[key]
    return None

def remove_editorial_boilerplate(value):
    """Remove publisher/license fragments that can contaminate topic terms."""
    text = clean_whitespace(value)
    if not text:
        return ""

    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    boilerplate_patterns = [
        r"\s*©\s*\d{4}.*$",
        r"\s*\(c\)\s*\d{4}.*$",
        r"\s*copyright\s+©?\s*\d{4}.*$",
        r"\s*all rights reserved\.?.*$",
        r"\s*this is an open access article.*$",
        r"\s*distributed under the terms of.*$",
        r"\s*creative commons.*$",
        r"\s*published by .*$",
        r"\s*publisher:? .*$",
    ]
    for pattern in boilerplate_patterns:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)

    text = re.sub(r"\s+", " ", text).strip()
    return text

def latex_escape(text):
    if pd.isna(text):
        return ""
    text = str(text)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

def add_strip_title(ax, title: str, fontsize: int = 10):
    ax.add_patch(plt.Rectangle(
        (0, 1.005), 1, 0.075,
        transform=ax.transAxes,
        facecolor=COLORS["gray_strip"],
        edgecolor="#888888",
        linewidth=0.7,
        clip_on=False,
    ))
    ax.text(
        0.5, 1.042, title,
        transform=ax.transAxes,
        ha="center",
        va="center",
        fontsize=fontsize,
        color=COLORS["text"],
    )

def polish_axis(ax):
    ax.grid(True, axis="both", color=COLORS["grid"], linewidth=0.65)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#A0A0A0")
    ax.spines["bottom"].set_color("#A0A0A0")

def save_figure(fig, name: str):
    png = OUTPUT_DIR / "figures" / f"{name}.png"
    pdf = OUTPUT_DIR / "figures" / f"{name}.pdf"
    fig.savefig(png)
    fig.savefig(pdf)
    print("Saved:", png)
    print("Saved:", pdf)
    return png, pdf

def safe_display(df, n=10):
    display(df.head(n))
    print("Rows:", len(df), "Columns:", len(df.columns))


## 4. Load and Validate the Scopus Corpus


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing corpus file: {DATA_PATH}")

df_raw = read_csv_robust(DATA_PATH)
thesaurus = (
    read_csv_robust(THESAURUS_PATH)
    if THESAURUS_PATH.exists()
    else pd.DataFrame(columns=["From", "To"])
)

colmap = {
    "title": find_column(df_raw, ["Title", "TI"]),
    "abstract": find_column(df_raw, ["Abstract", "AB"]),
    "year": find_column(df_raw, ["Year", "Publication Year", "PY"]),
    "source": find_column(df_raw, ["Source title", "Source", "SO", "Journal"]),
    "doi": find_column(df_raw, ["DOI", "DI"]),
    "author_keywords": find_column(df_raw, ["Author Keywords", "DE"]),
    "index_keywords": find_column(df_raw, ["Index Keywords", "Keywords Plus", "ID"]),
    "document_type": find_column(df_raw, ["Document Type", "DT"]),
    "language": find_column(df_raw, ["Language of Original Document", "Language", "LA"]),
}

abstract_col = colmap["abstract"]
title_col = colmap["title"]
year_col = colmap["year"]
source_col = colmap["source"]
doi_col = colmap["doi"]
author_kw_col = colmap["author_keywords"]
index_kw_col = colmap["index_keywords"]
doc_type_col = colmap["document_type"]
language_col = colmap["language"]

if abstract_col is None:
    raise ValueError("No abstract column was found. Expected 'Abstract' or 'AB'.")

print("Raw dataset shape:", df_raw.shape)
print("Column map:", colmap)
print("Thesaurus shape:", thesaurus.shape)

working = df_raw.copy()
working["abstract_clean"] = working[abstract_col].map(clean_whitespace)
working["title_clean"] = working[title_col].map(clean_whitespace) if title_col else ""
working["year_clean"] = working[year_col] if year_col else np.nan
working["source_clean"] = working[source_col].map(clean_whitespace) if source_col else ""
working["doi_clean"] = working[doi_col].map(clean_whitespace) if doi_col else ""
working["author_keywords_clean"] = working[author_kw_col].map(clean_whitespace) if author_kw_col else ""
working["index_keywords_clean"] = working[index_kw_col].map(clean_whitespace) if index_kw_col else ""
working["abstract_word_count"] = working["abstract_clean"].str.split().str.len().fillna(0).astype(int)

invalid_abstract = working["abstract_clean"].str.lower().str.strip().isin([
    "",
    "[no abstract available]",
    "no abstract available",
    "no abstract available.",
    "abstract not available",
    "not available",
])
short_abstract = working["abstract_word_count"] < 25
working["has_usable_abstract"] = ~invalid_abstract & ~short_abstract

quality_summary = pd.DataFrame([
    {"Metric": "Initial records", "Value": len(working)},
    {"Metric": "Records with empty/no abstract", "Value": int(invalid_abstract.sum())},
    {"Metric": "Records with very short abstract (<25 words)", "Value": int(short_abstract.sum())},
    {"Metric": "Records retained for abstract modeling", "Value": int(working["has_usable_abstract"].sum())},
    {"Metric": "Records with author keywords", "Value": int((working["author_keywords_clean"] != "").sum())},
    {"Metric": "Records with index keywords", "Value": int((working["index_keywords_clean"] != "").sum())},
    {"Metric": "Median abstract word count", "Value": int(working.loc[working["has_usable_abstract"], "abstract_word_count"].median())},
    {"Metric": "Mean abstract word count", "Value": round(float(working.loc[working["has_usable_abstract"], "abstract_word_count"].mean()), 2)},
])
quality_summary.to_csv(OUTPUT_DIR / "tables" / "abstract_modeling_quality_summary.csv", index=False)
quality_summary.to_csv(OUTPUT_DIR / "tables" / "abstract_corpus_quality_summary.csv", index=False)

topic_df = working.loc[working["has_usable_abstract"]].copy().reset_index(drop=True)
topic_df["doc_id"] = np.arange(len(topic_df))

excluded_cols = [
    "title_clean", "year_clean", "source_clean", "doi_clean",
    "abstract_clean", "abstract_word_count",
]
excluded_abstract_records = working.loc[
    ~working["has_usable_abstract"], excluded_cols
].copy()
excluded_abstract_records.to_csv(
    OUTPUT_DIR / "tables" / "abstract_records_excluded_by_text_filter.csv",
    index=False,
    encoding="utf-8-sig",
)

export_cols = [
    "doc_id", "title_clean", "year_clean", "source_clean", "doi_clean",
    "abstract_clean", "author_keywords_clean", "index_keywords_clean",
]
topic_df[export_cols].to_csv(
    OUTPUT_DIR / "tables" / "abstract_texts_used_for_bertopic.csv",
    index=False,
    encoding="utf-8-sig",
)

safe_display(quality_summary, 10)
print("Documents modeled:", len(topic_df))


### Methodological note

The topic model is intentionally restricted to abstracts. Abstracts provide a stable and comparable textual unit across the Scopus records because they summarize objectives, methods, context, and findings. Titles and keywords are retained as metadata for interpretation and validation, but they are not concatenated with the abstracts for embedding generation.

Before embedding generation, abstracts are only minimally cleaned: encoding and whitespace are normalized, and records without usable abstract text are flagged. More intensive text normalization is intentionally postponed until the topic representation stage, after clustering, in order to preserve the natural semantic form of the abstracts for transformer-based embeddings.


## 5. Representation Preprocessing


BERTopic uses two textual tracks in this notebook. The embedding stage receives minimally cleaned abstracts in their natural linguistic form, because transformer models are trained on full text and benefit from syntactic and contextual information. After clusters have been identified, the c-TF-IDF representation stage receives a more normalized version of the abstracts. At this point, editorial boilerplate, URLs, generic academic wording, stopwords, and inflectional variation are reduced so that the words used to describe topics are cleaner and easier to interpret.


In [ ]:
custom_stopwords = set(ENGLISH_STOP_WORDS)
custom_stopwords.update({
    "study", "article", "paper", "research", "result", "results",
    "finding", "findings", "method", "methods", "data", "based",
    "using", "used", "show", "shows", "suggest", "suggests",
    "examine", "examines", "examined", "approach", "case",
    "aim", "aims", "purpose", "paper", "article"
})

def load_spacy_model():
    try:
        import spacy
    except Exception:
        print("spaCy is not installed. Falling back to rule-based cleaning.")
        return None

    try:
        return spacy.load("en_core_web_sm", disable=["ner", "parser"])
    except Exception:
        print("spaCy model en_core_web_sm not found. Attempting automatic download...")
        try:
            subprocess.run(
                [sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
                check=False,
            )
            return spacy.load("en_core_web_sm", disable=["ner", "parser"])
        except Exception as exc:
            print("Could not load en_core_web_sm. Falling back to rule-based cleaning.")
            print("Reason:", repr(exc))
            return None

nlp = load_spacy_model()
print("spaCy lemmatization active:", nlp is not None)

def normalize_for_representation(text: str) -> str:
    text = remove_editorial_boilerplate(text)
    text = clean_whitespace(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if nlp is not None:
        doc = nlp(text)
        tokens = [
            tok.lemma_.lower().strip()
            for tok in doc
            if tok.is_alpha and len(tok.lemma_) > 2 and tok.lemma_.lower() not in custom_stopwords
        ]
    else:
        tokens = [
            tok for tok in text.split()
            if len(tok) > 2 and tok not in custom_stopwords
        ]
    return " ".join(tokens)

topic_df["abstract_for_representation"] = topic_df["abstract_clean"].map(remove_editorial_boilerplate)
topic_df["representation_boilerplate_removed"] = (
    topic_df["abstract_clean"].str.len()
    .ne(topic_df["abstract_for_representation"].str.len())
)

docs_embedding = topic_df["abstract_clean"].tolist()
docs_representation = topic_df["abstract_for_representation"].map(normalize_for_representation).tolist()
topic_df["bertopic_text"] = docs_representation

representation_summary = pd.DataFrame([
    {"Metric": "Documents sent to embedding stage", "Value": len(docs_embedding)},
    {"Metric": "Documents sent to c-TF-IDF representation stage", "Value": len(docs_representation)},
    {"Metric": "Documents with editorial boilerplate reduced before c-TF-IDF", "Value": int(topic_df["representation_boilerplate_removed"].sum())},
    {"Metric": "spaCy lemmatization active", "Value": bool(nlp is not None)},
])
representation_summary.to_csv(
    OUTPUT_DIR / "tables" / "representation_preprocessing_summary.csv",
    index=False,
)

preprocess_example = pd.DataFrame({
    "Stage": [
        "Minimal abstract used for embeddings",
        "Abstract after representation-only boilerplate removal",
        "Final c-TF-IDF representation text",
    ],
    "Example": [
        topic_df.loc[0, "abstract_clean"][:260],
        topic_df.loc[0, "abstract_for_representation"][:260],
        topic_df.loc[0, "bertopic_text"][:260],
    ],
})
preprocess_example.to_csv(OUTPUT_DIR / "tables" / "representation_preprocessing_example.csv", index=False)
display(preprocess_example)


## 6. Embedding Robustness Check and Hyperparameter Grid


In [ ]:
# The grid is intentionally small. It is not meant to benchmark every possible
# BERTopic setting; it documents that the final configuration was not selected blindly.

EMBEDDING_CANDIDATES = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
]

EMBEDDING_BATCH_SIZE = {
    "all-MiniLM-L6-v2": 32,
    "all-mpnet-base-v2": 8,
}

GRID = [
    {"n_neighbors": 10, "n_components": 5, "min_dist": 0.0,  "min_cluster_size": 15, "min_samples": 5},
    {"n_neighbors": 15, "n_components": 5, "min_dist": 0.0,  "min_cluster_size": 15, "min_samples": 5},
    {"n_neighbors": 15, "n_components": 5, "min_dist": 0.05, "min_cluster_size": 20, "min_samples": 5},
    {"n_neighbors": 30, "n_components": 5, "min_dist": 0.05, "min_cluster_size": 20, "min_samples": 10},
]

VECTOR_PARAMS = {
    "ngram_range": (1, 3),
    "min_df": 5,
    "max_df": 0.70,
}

TOP_N_WORDS = 10
DEVICE = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"

print("Device:", DEVICE)
print("Embedding candidates:", EMBEDDING_CANDIDATES)
print("Grid size:", len(EMBEDDING_CANDIDATES) * len(GRID))


In [ ]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

def compute_topic_diversity(model: BERTopic, top_n: int = 10) -> float:
    words = []
    for tid in sorted([t for t in model.get_topics().keys() if t != -1]):
        topic_terms = model.get_topic(tid) or []
        words.extend([w for w, _ in topic_terms[:top_n]])
    if not words:
        return np.nan
    return len(set(words)) / len(words)

def compute_cv_coherence(model: BERTopic, docs: list[str], top_n: int = 10) -> float:
    topic_ids = sorted([t for t in model.get_topics().keys() if t != -1])
    topic_words = []
    for tid in topic_ids:
        terms = model.get_topic(tid) or []
        topic_words.append([w for w, _ in terms[:top_n]])
    tokenized_docs = [doc.split() for doc in docs]
    dictionary = Dictionary(tokenized_docs)
    dictionary.filter_extremes(no_below=2, no_above=0.90)
    if not topic_words or len(dictionary) == 0:
        return np.nan
    cm = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence="c_v",
        processes=1,
    )
    return float(cm.get_coherence())

def compute_silhouette(reduced_embeddings, labels) -> float:
    labels = np.array(labels)
    mask = labels != -1
    if mask.sum() < 10 or len(set(labels[mask])) < 2:
        return np.nan
    sample_size = min(1000, int(mask.sum()))
    return float(silhouette_score(
        reduced_embeddings[mask],
        labels[mask],
        metric="euclidean",
        sample_size=sample_size,
        random_state=SEED,
    ))

def compute_davies_bouldin(reduced_embeddings, labels) -> float:
    labels = np.array(labels)
    mask = labels != -1
    if mask.sum() < 10 or len(set(labels[mask])) < 2:
        return np.nan
    return float(davies_bouldin_score(reduced_embeddings[mask], labels[mask]))

def make_vectorizer():
    return CountVectorizer(
        stop_words=list(custom_stopwords),
        ngram_range=VECTOR_PARAMS["ngram_range"],
        min_df=VECTOR_PARAMS["min_df"],
        max_df=VECTOR_PARAMS["max_df"],
    )

def train_bertopic_for_config(embedding_name: str, params: dict, embeddings: np.ndarray):
    umap_model = UMAP(
        n_neighbors=params["n_neighbors"],
        n_components=params["n_components"],
        min_dist=params["min_dist"],
        metric="cosine",
        random_state=SEED,
        transform_seed=SEED,
        n_jobs=1,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=params["min_cluster_size"],
        min_samples=params["min_samples"],
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
        core_dist_n_jobs=1,
        approx_min_span_tree=False,
        gen_min_span_tree=True,
    )
    topic_model = BERTopic(
        embedding_model=None,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=make_vectorizer(),
        ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
        top_n_words=TOP_N_WORDS,
        calculate_probabilities=True,
        verbose=False,
    )
    topics, probs = topic_model.fit_transform(docs_representation, embeddings=embeddings)
    return topic_model, topics, probs

embedding_cache = {}
grid_rows = []
model_cache = {}

GRID_RESULTS_PATH = OUTPUT_DIR / "grid" / f"embedding_hyperparameter_grid_results_{PREPROCESSING_VERSION}.csv"
USE_EXISTING_GRID = GRID_RESULTS_PATH.exists()
RUN_GRID = not USE_EXISTING_GRID

if RUN_GRID:
    for embedding_name in EMBEDDING_CANDIDATES:
        emb_path = OUTPUT_DIR / "models" / f"embeddings_{embedding_name.replace('/', '_')}_seed{SEED}_{PREPROCESSING_VERSION}.npy"
        if emb_path.exists():
            embeddings = np.load(emb_path)
        else:
            encoder = SentenceTransformer(embedding_name, device=DEVICE)
            embeddings = encoder.encode(
                docs_embedding,
                show_progress_bar=True,
                batch_size=EMBEDDING_BATCH_SIZE.get(embedding_name, 16),
                normalize_embeddings=True,
            )
            np.save(emb_path, embeddings)
        embedding_cache[embedding_name] = embeddings

        for i, params in enumerate(GRID, start=1):
            run_id = f"{embedding_name.split('/')[-1]}_g{i}"
            print("Running", run_id, params)
            topic_model, topics, probs = train_bertopic_for_config(embedding_name, params, embeddings)
            labels = np.array(topics)
            n_docs = len(labels)
            n_outliers = int((labels == -1).sum())
            n_topics = len([t for t in set(labels) if t != -1])
            diversity = compute_topic_diversity(topic_model, top_n=10)
            try:
                coherence = compute_cv_coherence(topic_model, docs_representation, top_n=10)
            except Exception as exc:
                print("Coherence failed for", run_id, exc)
                coherence = np.nan
            try:
                reduced = topic_model.umap_model.embedding_
                silhouette = compute_silhouette(reduced, labels)
            except Exception as exc:
                print("Silhouette failed for", run_id, exc)
                silhouette = np.nan
            try:
                reduced = topic_model.umap_model.embedding_
                davies_bouldin = compute_davies_bouldin(reduced, labels)
            except Exception as exc:
                print("Davies-Bouldin failed for", run_id, exc)
                davies_bouldin = np.nan

            row = {
                "run_id": run_id,
                "embedding_model": embedding_name,
                **params,
                "topics": n_topics,
                "outliers": n_outliers,
                "outlier_pct": round(n_outliers / n_docs * 100, 2),
                "topic_diversity": round(diversity, 4),
                "coherence_cv": round(coherence, 4) if not np.isnan(coherence) else np.nan,
                "silhouette": round(silhouette, 4) if not np.isnan(silhouette) else np.nan,
                "davies_bouldin": round(davies_bouldin, 4) if not np.isnan(davies_bouldin) else np.nan,
            }
            grid_rows.append(row)
            model_cache[run_id] = {
                "model": topic_model,
                "topics": topics,
                "probabilities": probs,
                "embeddings": embeddings,
            }

    grid_results = pd.DataFrame(grid_rows)

    # Composite score used only for ranking candidates; final selection still
    # requires qualitative inspection.
    scored = grid_results.copy()
    for metric in ["coherence_cv", "topic_diversity", "silhouette"]:
        vals = scored[metric].astype(float)
        if vals.notna().sum() > 1:
            scored[f"{metric}_scaled"] = minmax_scale(vals.fillna(vals.min()))
        else:
            scored[f"{metric}_scaled"] = 0.0
    vals = scored["davies_bouldin"].astype(float)
    if vals.notna().sum() > 1:
        scored["davies_bouldin_scaled_inverse"] = 1 - minmax_scale(vals.fillna(vals.max()))
    else:
        scored["davies_bouldin_scaled_inverse"] = 0.0
    scored["outlier_scaled"] = 1 - (scored["outlier_pct"] / 100).clip(0, 1)
    scored["topic_count_penalty"] = scored["topics"].apply(lambda x: 0 if 12 <= x <= 40 else 0.15)
    scored["selection_score"] = (
        0.35 * scored["coherence_cv_scaled"]
        + 0.25 * scored["topic_diversity_scaled"]
        + 0.15 * scored["silhouette_scaled"]
        + 0.05 * scored["davies_bouldin_scaled_inverse"]
        + 0.20 * scored["outlier_scaled"]
        - scored["topic_count_penalty"]
    )
    grid_results = scored.sort_values("selection_score", ascending=False).reset_index(drop=True)
    grid_results.to_csv(GRID_RESULTS_PATH, index=False)
else:
    print("Using existing grid results:", GRID_RESULTS_PATH)
    grid_results = pd.read_csv(GRID_RESULTS_PATH)

display(grid_results)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.8))

ax = axes[0]
plot_df = grid_results.sort_values("selection_score", ascending=True)
y_labels = plot_df["run_id"]
ax.barh(y_labels, plot_df["selection_score"], color=COLORS["teal"], edgecolor=COLORS["teal_dark"], linewidth=0.6)
ax.set_xlabel("Selection score")
ax.set_ylabel("")
add_strip_title(ax, "Embedding and hyperparameter grid ranking", fontsize=9)
polish_axis(ax)

ax = axes[1]
sns.scatterplot(
    data=grid_results,
    x="outlier_pct",
    y="coherence_cv",
    hue="embedding_model",
    size="topics",
    sizes=(50, 250),
    palette=[COLORS["teal"], COLORS["brown"]],
    edgecolor="#555555",
    linewidth=0.5,
    ax=ax,
)
ax.set_xlabel("Outlier documents (%)")
ax.set_ylabel("Topic coherence $c_v$")
add_strip_title(ax, "Coherence vs. outlier ratio", fontsize=9)
polish_axis(ax)
ax.legend(frameon=True, fontsize=7, loc="best")

fig.suptitle("BERTopic robustness check for embedding and clustering choices", fontsize=13, fontweight="bold", y=1.03)
fig.tight_layout()
save_figure(fig, "embedding_hyperparameter_grid_summary")
show_figure(fig)


## 7. Select Final Configuration

The top-ranked configuration is selected automatically as a starting point. Before final reporting, inspect the resulting topics and representative abstracts. If a different configuration is selected after expert review, set `SELECTED_RUN_ID` manually.


In [ ]:
SELECTED_RUN_ID = None  # Example: "all-mpnet-base-v2_g2"; leave as None to use top-ranked grid result.

if SELECTED_RUN_ID is None:
    SELECTED_RUN_ID = grid_results.iloc[0]["run_id"]

print("Selected run:", SELECTED_RUN_ID)
selected_meta = grid_results.loc[grid_results["run_id"] == SELECTED_RUN_ID].iloc[0].to_dict()
print(json.dumps(selected_meta, indent=2))

if SELECTED_RUN_ID in model_cache:
    selected = model_cache[SELECTED_RUN_ID]
    topic_model = selected["model"]
    topics = selected["topics"]
    probabilities = selected["probabilities"]
    embeddings = selected["embeddings"]
else:
    # Re-train selected run if the notebook was restarted after the grid.
    embedding_name = selected_meta["embedding_model"]
    params = {k: selected_meta[k] for k in ["n_neighbors", "n_components", "min_dist", "min_cluster_size", "min_samples"]}
    emb_path = OUTPUT_DIR / "models" / f"embeddings_{embedding_name.replace('/', '_')}_seed{SEED}_{PREPROCESSING_VERSION}.npy"
    if emb_path.exists():
        embeddings = np.load(emb_path)
    else:
        encoder = SentenceTransformer(embedding_name, device=DEVICE)
        embeddings = encoder.encode(
            docs_embedding,
            show_progress_bar=True,
            batch_size=EMBEDDING_BATCH_SIZE.get(embedding_name, 16),
            normalize_embeddings=True,
        )
        np.save(emb_path, embeddings)
    topic_model, topics, probabilities = train_bertopic_for_config(embedding_name, params, embeddings)

topic_df["topic"] = topics
if probabilities is not None:
    if isinstance(probabilities, np.ndarray) and probabilities.ndim == 2:
        topic_df["topic_probability"] = probabilities.max(axis=1)
    else:
        topic_df["topic_probability"] = probabilities
else:
    topic_df["topic_probability"] = np.nan

topic_info = topic_model.get_topic_info()
topic_info.to_csv(OUTPUT_DIR / "tables" / "selected_topic_info_raw.csv", index=False)
display(topic_info.head(15))


## 8. Topic Inventory and Labeling Evidence


In [ ]:
def get_terms(topic_id: int, top_n: int = 8):
    terms = topic_model.get_topic(topic_id) or []
    return "; ".join([term for term, score in terms[:top_n]])

topic_ids = sorted([int(t) for t in topic_info["Topic"].tolist() if int(t) != -1])
inventory_rows = []
evidence_rows = []

for tid in topic_ids:
    subset = topic_df[topic_df["topic"] == tid].copy()
    n = len(subset)
    pct = round(n / len(topic_df) * 100, 2)
    terms = get_terms(tid, top_n=8)
    label_seed = " / ".join([term.strip() for term in terms.split(";")[:3]])
    inventory_rows.append({
        "Topic": tid,
        "Provisional label": label_seed,
        "Documents": n,
        "Documents (%)": pct,
        "Representative terms": terms,
    })

    subset = subset.sort_values("topic_probability", ascending=False)
    for rank, (_, row) in enumerate(subset.head(5).iterrows(), start=1):
        evidence_rows.append({
            "Topic": tid,
            "Rank": rank,
            "Topic probability": row.get("topic_probability", np.nan),
            "Title": row.get("title_clean", ""),
            "Abstract": row.get("abstract_clean", ""),
            "Representative terms": terms,
        })

topic_table = pd.DataFrame(inventory_rows)
labeling_evidence = pd.DataFrame(evidence_rows)

topic_table.to_csv(OUTPUT_DIR / "tables" / "selected_topic_inventory_provisional.csv", index=False)
labeling_evidence.to_csv(OUTPUT_DIR / "labeling_evidence" / "representative_abstracts_for_labeling.csv", index=False)

display(topic_table)


### How topic labels should be validated

The provisional labels above are not final thesis labels. For each topic, review:

1. the c-TF-IDF representative terms;
2. the five highest-probability abstracts;
3. the thematic consistency across those abstracts;
4. whether the label introduces concepts not supported by the evidence;
5. whether the label is too broad, too narrow, or too close to another topic.


## 9. Local Llama 3.1-Assisted Labeling (No API)

This section provides a reproducible alternative to the OpenAI-assisted labeling stage. It uses a local Ollama server and the `llama3.1:8b` model. If Ollama is not installed or the model is not available, the notebook does not fail: it exports the prompts and keeps the c-TF-IDF provisional labels.

To run this section locally, install Ollama, start the Ollama server, and pull the model:

```bash
ollama serve
ollama pull llama3.1:8b
```

The labeling request uses structured JSON output so the notebook can reliably extract the proposed topic label and rationale. The generated labels remain suggestions and should be validated manually against the representative terms and abstracts.


In [ ]:
# ============================================================
# Local Llama 3.1-assisted topic labeling via Ollama
# ============================================================

import requests
import subprocess
import time

LLAMA_MODEL = "llama3.1:8b"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
USE_LOCAL_LLAMA_LABELING = True
AUTO_START_OLLAMA = True       # Tries to start `ollama serve` if Ollama is installed.
AUTO_PULL_LLAMA_MODEL = False  # Set True only if you want the notebook to download llama3.1:8b.

LLAMA_LABEL_SCHEMA = {
    "type": "object",
    "properties": {
        "topic": {
            "type": "string",
            "description": "A concise academic label for the topic, between 4 and 8 words."
        },
        "rationale": {
            "type": "string",
            "description": "One concise sentence explaining why the label fits the evidence."
        }
    },
    "required": ["topic", "rationale"]
}

LLAMA_PROMPT_TEMPLATE = """
You are assisting with academic topic labeling for a master's thesis about
storytelling, narratives, and news media.

Use only the evidence provided below. Do not introduce concepts that are not
supported by the representative terms, titles, or abstracts.

Your task is to transform the statistical topic representation into a concise
academic label. Do not simply copy the first three terms. Use the terms and the
representative abstracts together.

Label requirements:
- 4 to 8 words.
- Academic and interpretable.
- No slash-separated labels.
- Do not use "Topic" in the label.
- Avoid overly generic labels such as "News Media Narratives" unless the evidence is truly broad.

Topic ID: {topic_id}

Representative c-TF-IDF terms:
{terms}

Representative abstracts:
{documents}

Return valid JSON only with this structure:
{{
  "topic": "<short academic label>",
  "rationale": "<one concise sentence explaining the label>"
}}
"""

for subdir in ["labeling_evidence", "tables"]:
    (OUTPUT_DIR / subdir).mkdir(parents=True, exist_ok=True)

(OUTPUT_DIR / "labeling_evidence" / "llama31_prompt_template.txt").write_text(
    LLAMA_PROMPT_TEMPLATE.strip(),
    encoding="utf-8",
)


def build_topic_prompt(topic_id: int, max_chars_per_doc: int = 650, max_docs: int = 4) -> str:
    terms = topic_table.loc[
        topic_table["Topic"] == topic_id,
        "Representative terms",
    ].iloc[0]

    docs = labeling_evidence[labeling_evidence["Topic"] == topic_id].head(max_docs)
    doc_text = []
    for _, row in docs.iterrows():
        title = clean_whitespace(row.get("Title", ""))
        abstract = clean_whitespace(row.get("Abstract", ""))[:max_chars_per_doc]
        doc_text.append(f"- Title: {title}\n  Abstract: {abstract}")

    return LLAMA_PROMPT_TEMPLATE.format(
        topic_id=topic_id,
        terms=terms,
        documents="\n".join(doc_text),
    )


def parse_llama_topic_response(text: str) -> tuple[str, str]:
    """Parse Llama JSON output, with a fallback for imperfect responses."""
    normalized = text.strip()

    try:
        parsed = json.loads(normalized)
        label = clean_whitespace(parsed.get("topic", ""))
        rationale = clean_whitespace(parsed.get("rationale", ""))
        if label:
            return label, rationale or "Label generated from representative c-TF-IDF terms and abstracts."
    except Exception:
        pass

    # Fallback parser for non-JSON responses.
    label = ""
    rationale = ""
    for raw_line in normalized.splitlines():
        line = raw_line.strip().strip("{} ,")
        if line.lower().startswith("topic"):
            label = line.split(":", 1)[1].strip().strip("\"',") if ":" in line else label
        elif line.lower().startswith("rationale"):
            rationale = line.split(":", 1)[1].strip().strip("\"',") if ":" in line else rationale

    if not label:
        label = clean_whitespace(normalized).split("\n")[0][:90]
    if not rationale:
        rationale = "Label generated from representative c-TF-IDF terms and abstracts."

    label = re.sub(r"^[\"']|[\"']$", "", label).strip()
    return label, rationale


def label_needs_review(label: str, provisional: str) -> bool:
    label_clean = clean_whitespace(label).lower()
    provisional_clean = clean_whitespace(provisional).lower()
    if not label_clean:
        return True
    if "/" in label_clean:
        return True
    if label_clean == provisional_clean:
        return True
    if len(label_clean.split()) > 10:
        return True
    if label_clean in {"topic", "news media", "news media narratives"}:
        return True
    return False


def ollama_is_running() -> bool:
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=3)
        return response.status_code == 200
    except Exception:
        return False


def start_ollama_if_possible() -> bool:
    if ollama_is_running():
        return True
    if not AUTO_START_OLLAMA:
        return False
    if shutil.which("ollama") is None:
        return False

    print("Starting local Ollama server...")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    for _ in range(20):
        time.sleep(1)
        if ollama_is_running():
            return True
    return False


def ollama_model_available(model_name: str) -> bool:
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        response.raise_for_status()
        models = response.json().get("models", [])
        names = {m.get("name", "") for m in models}
        return model_name in names or model_name.split(":")[0] in {n.split(":")[0] for n in names}
    except Exception:
        return False


def pull_ollama_model(model_name: str) -> bool:
    if not AUTO_PULL_LLAMA_MODEL:
        return False
    if shutil.which("ollama") is None:
        return False
    print(f"Pulling {model_name}. This may take several minutes...")
    result = subprocess.run(["ollama", "pull", model_name], check=False)
    return result.returncode == 0


def llama_chat(prompt: str) -> str:
    payload = {
        "model": LLAMA_MODEL,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You label BERTopic clusters for academic research. "
                    "Base the label only on the evidence provided. "
                    "Return valid JSON only."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        "stream": False,
        "format": LLAMA_LABEL_SCHEMA,
        "options": {
            "temperature": 0.0,
            "top_p": 0.9,
            "num_predict": 140,
        },
    }
    response = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json=payload, timeout=240)
    response.raise_for_status()
    return response.json()["message"]["content"].strip()


# Always keep a final-label column so downstream figures work with or without Llama.
topic_table["Final label"] = topic_table["Provisional label"]
topic_table["Llama rationale"] = ""
topic_table["Label source"] = "c-TF-IDF provisional"
topic_table["Needs manual review"] = True
LABEL_COLUMN = "Final label"

# Export prompt examples for methodological transparency.
prompt_examples = [
    {"Topic": tid, "Prompt": build_topic_prompt(tid)}
    for tid in topic_ids[:3]
]
pd.DataFrame(prompt_examples).to_csv(
    OUTPUT_DIR / "labeling_evidence" / "llama31_prompt_examples.csv",
    index=False,
    encoding="utf-8-sig",
)

llama_ready = False
if USE_LOCAL_LLAMA_LABELING:
    if start_ollama_if_possible():
        if ollama_model_available(LLAMA_MODEL) or pull_ollama_model(LLAMA_MODEL):
            llama_ready = True
        else:
            print(f"Ollama is running, but {LLAMA_MODEL} is not available.")
            print(f"Run: ollama pull {LLAMA_MODEL}")
    else:
        print("Ollama is not running or not installed.")
        print("Install/start Ollama, then run: ollama pull llama3.1:8b")

llama_results = []
if USE_LOCAL_LLAMA_LABELING and llama_ready:
    print(f"Local Llama labeling enabled with model: {LLAMA_MODEL}")

    for tid in topic_ids:
        prompt = build_topic_prompt(tid)
        provisional = topic_table.loc[topic_table["Topic"] == tid, "Provisional label"].iloc[0]
        try:
            response_text = llama_chat(prompt)
            label, rationale = parse_llama_topic_response(response_text)
            status = "ok"
        except Exception as exc:
            response_text = ""
            label = provisional
            rationale = f"Llama labeling failed; provisional label retained. Error: {repr(exc)}"
            status = "error"

        needs_review = label_needs_review(label, provisional)
        topic_table.loc[topic_table["Topic"] == tid, "Final label"] = label
        topic_table.loc[topic_table["Topic"] == tid, "Llama rationale"] = rationale
        topic_table.loc[topic_table["Topic"] == tid, "Label source"] = (
            "Llama 3.1-assisted" if status == "ok" else "c-TF-IDF provisional"
        )
        topic_table.loc[topic_table["Topic"] == tid, "Needs manual review"] = needs_review

        llama_results.append({
            "Topic": tid,
            "Llama model": LLAMA_MODEL,
            "Status": status,
            "Prompt": prompt,
            "Llama response": response_text,
            "Parsed label": label,
            "Parsed rationale": rationale,
            "Needs manual review": needs_review,
        })

    llama_df = pd.DataFrame(llama_results)
    llama_df.to_csv(
        OUTPUT_DIR / "labeling_evidence" / "llama31_suggested_topic_labels.csv",
        index=False,
        encoding="utf-8-sig",
    )

    topic_table.to_csv(
        OUTPUT_DIR / "tables" / "selected_topic_inventory_with_llama31_labels.csv",
        index=False,
        encoding="utf-8-sig",
    )
    topic_table.to_csv(
        OUTPUT_DIR / "tables" / "selected_topic_inventory_with_final_labels.csv",
        index=False,
        encoding="utf-8-sig",
    )

    display(llama_df[["Topic", "Status", "Parsed label", "Parsed rationale", "Needs manual review"]])
    display(topic_table[["Topic", "Final label", "Documents", "Documents (%)", "Representative terms", "Needs manual review"]])

else:
    print("Local Llama labeling was not executed.")
    print("Prompt examples were exported, and c-TF-IDF provisional labels were retained.")
    topic_table.to_csv(
        OUTPUT_DIR / "tables" / "selected_topic_inventory_with_llama31_labels.csv",
        index=False,
        encoding="utf-8-sig",
    )
    topic_table.to_csv(
        OUTPUT_DIR / "tables" / "selected_topic_inventory_with_final_labels.csv",
        index=False,
        encoding="utf-8-sig",
    )
    display(topic_table[["Topic", "Final label", "Documents", "Documents (%)", "Representative terms", "Needs manual review"]])


## 10. Model Diagnostics


In [ ]:
labels = np.array(topic_df["topic"])
n_documents = len(topic_df)
n_substantive_topics = len(topic_ids)
n_outliers = int((labels == -1).sum())
outlier_ratio = round((n_outliers / n_documents) * 100, 2)
diversity_10 = round(compute_topic_diversity(topic_model, top_n=10), 4)
coherence_cv = round(compute_cv_coherence(topic_model, docs_representation, top_n=10), 4)
silhouette = round(compute_silhouette(topic_model.umap_model.embedding_, labels), 4)
davies_bouldin = round(compute_davies_bouldin(topic_model.umap_model.embedding_, labels), 4)

diagnostics = pd.DataFrame({
    "Metric": [
        "Documents modeled",
        "Substantive topics",
        "Outlier documents",
        "Outlier documents (%)",
        "Topic diversity (top 10 terms)",
        "Topic coherence c_v",
        "Silhouette score",
        "Davies-Bouldin index",
        "Embedding model",
        "UMAP n_neighbors",
        "UMAP n_components",
        "UMAP min_dist",
        "HDBSCAN min_cluster_size",
        "HDBSCAN min_samples",
        "Random seed",
    ],
    "Value": [
        n_documents,
        n_substantive_topics,
        n_outliers,
        outlier_ratio,
        diversity_10,
        coherence_cv,
        silhouette,
        davies_bouldin,
        selected_meta["embedding_model"],
        selected_meta["n_neighbors"],
        selected_meta["n_components"],
        selected_meta["min_dist"],
        selected_meta["min_cluster_size"],
        selected_meta["min_samples"],
        SEED,
    ],
})
diagnostics.to_csv(OUTPUT_DIR / "tables" / "selected_model_diagnostics.csv", index=False)
display(diagnostics)


## 11. Figures


In [ ]:
# Topic prevalence
plot_df = topic_table.sort_values("Documents", ascending=True).copy()
prevalence_labels = [
    f"T{int(t)}: {textwrap.shorten(str(lab), width=42, placeholder='...')}"
    for t, lab in zip(plot_df["Topic"], plot_df[LABEL_COLUMN])
]
fig, ax = plt.subplots(figsize=(10.6, max(5.6, 0.29 * len(plot_df))))
ax.barh(
    prevalence_labels,
    plot_df["Documents (%)"],
    color=COLORS["teal"],
    edgecolor=COLORS["teal_dark"],
    linewidth=0.6,
)
for y, val in enumerate(plot_df["Documents (%)"]):
    ax.text(val + 0.15, y, f"{val:.1f}%", va="center", ha="left", fontsize=7.5, color=COLORS["text"])
ax.set_xlabel("Share of abstract corpus (%)", fontsize=9)
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=8.3)
ax.tick_params(axis="x", labelsize=8)
add_strip_title(ax, "BERTopic topic prevalence", fontsize=10)
polish_axis(ax)
fig.tight_layout()
save_figure(fig, "selected_topic_prevalence")
show_figure(fig)


In [ ]:
# Representative c-TF-IDF terms for main topics
def make_topic_term_barchart(model: BERTopic, table: pd.DataFrame, top_topics: int = 12, top_terms: int = 6):
    selected = table.sort_values("Documents", ascending=False).head(top_topics).copy()
    n_rows = math.ceil(len(selected) / 3)
    fig, axes = plt.subplots(n_rows, 3, figsize=(13.2, 3.0 * n_rows), sharex=False)
    axes = np.array(axes).flatten()

    shades = ["#5E9999", "#6FA8A8", "#7FB6B6", "#90C1C1", "#A2CCCC", "#B5D7D7"]

    for ax, (_, row) in zip(axes, selected.iterrows()):
        tid = int(row["Topic"])
        terms_scores = (model.get_topic(tid) or [])[:top_terms]
        terms = [t for t, s in terms_scores][::-1]
        scores = [s for t, s in terms_scores][::-1]
        colors = shades[:len(scores)][::-1]
        ax.barh(terms, scores, color=colors, edgecolor=COLORS["teal_dark"], linewidth=0.5)
        ax.set_title(f"T{tid}. {row[LABEL_COLUMN]}", fontsize=8.2, color=COLORS["text"], pad=6, loc="left")
        ax.tick_params(axis="y", labelsize=7.2)
        ax.tick_params(axis="x", labelsize=7)
        ax.set_xlabel("c-TF-IDF", fontsize=7.5)
        polish_axis(ax)

    for ax in axes[len(selected):]:
        ax.axis("off")

    fig.suptitle("Representative c-TF-IDF terms for the main BERTopic clusters", fontsize=13, fontweight="bold", y=1.01)
    fig.tight_layout()
    save_figure(fig, "selected_representative_ctfidf_terms")
    return fig

fig = make_topic_term_barchart(topic_model, topic_table, top_topics=12, top_terms=6)
show_figure(fig)


In [ ]:
# 2D UMAP map for visualization only
umap_2d = UMAP(
    n_neighbors=int(selected_meta["n_neighbors"]),
    n_components=2,
    min_dist=0.05,
    metric="cosine",
    random_state=SEED,
    transform_seed=SEED,
    n_jobs=1,
)
coords = umap_2d.fit_transform(embeddings)
topic_df["x"] = coords[:, 0]
topic_df["y"] = coords[:, 1]

palette = [
    "#4F8F91", "#6FA8A8", "#8EBDBD", "#B7D5D5",
    "#6D8FB3", "#8CA9C8", "#B2C6DC",
    "#8A6B4E", "#A98461", "#C49E74",
    "#C7A64A", "#D8BC68", "#E5CF8A",
    "#7FAE7F", "#9DC49D", "#B9D8B9",
    "#9B82B7", "#B29BC8", "#C9B9DA",
]
topic_color = {tid: palette[i % len(palette)] for i, tid in enumerate(topic_ids)}

fig, ax = plt.subplots(figsize=(13.5, 9.5))
out = topic_df["topic"] == -1
ax.scatter(topic_df.loc[out, "x"], topic_df.loc[out, "y"], s=5, c="#BDBDBD", alpha=0.18, linewidths=0, label="Outlier documents")

for tid in topic_ids:
    subset = topic_df[topic_df["topic"] == tid]
    ax.scatter(
        subset["x"], subset["y"],
        s=14,
        c=topic_color[tid],
        alpha=0.42,
        linewidths=0,
    )

texts = []
for tid in topic_ids:
    subset = topic_df[topic_df["topic"] == tid]
    if subset.empty:
        continue
    cx, cy = subset[["x", "y"]].median()
    label = topic_table.loc[topic_table["Topic"] == tid, LABEL_COLUMN].iloc[0]
    short = "\n".join(textwrap.wrap(label, width=23))
    txt = ax.text(
        cx, cy, short,
        fontsize=8.2,
        color=topic_color[tid],
        ha="center",
        va="center",
        path_effects=[pe.withStroke(linewidth=3.0, foreground="white", alpha=0.92)],
    )
    texts.append(txt)

if adjust_text is not None:
    adjust_text(
        texts,
        ax=ax,
        arrowprops=dict(arrowstyle="-", color="#777777", lw=0.45, alpha=0.45),
        expand_points=(1.08, 1.18),
        expand_text=(1.05, 1.20),
        force_text=(0.06, 0.14),
    )

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_color("#9A9A9A")
    spine.set_linewidth(0.7)

ax.set_title("BERTopic map of abstract-level narrative themes", loc="left", fontsize=17, fontweight="bold", pad=22)
ax.text(
    0.0, 1.015,
    "Abstract embeddings projected with UMAP; labels require validation from c-TF-IDF terms and representative abstracts",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=10,
    color=COLORS["muted"],
)
legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=COLORS["teal"], markersize=7, label="Substantive topics"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#BDBDBD", markersize=7, label="Outlier documents"),
]
ax.legend(handles=legend_handles, frameon=True, fontsize=8, loc="lower right")
fig.tight_layout()
save_figure(fig, "selected_bertopic_umap_topic_map")
show_figure(fig)


## 12. Outlier Review


In [ ]:
outliers = topic_df[topic_df["topic"] == -1].copy()
outlier_cols = ["title_clean", "abstract_clean", "abstract_word_count"]
if year_col and year_col in topic_df.columns:
    outliers["Year"] = topic_df.loc[outliers.index, year_col].values
    outlier_cols.insert(0, "Year")
if doi_col and doi_col in topic_df.columns:
    outliers["DOI"] = topic_df.loc[outliers.index, doi_col].values
    outlier_cols.insert(1 if "Year" in outlier_cols else 0, "DOI")

outliers[outlier_cols].to_csv(OUTPUT_DIR / "outliers" / "outlier_documents_for_review.csv", index=False)
print("Outliers:", len(outliers))
display(outliers[outlier_cols].head(20))


## 13. LaTeX and Methodology Snippets


In [ ]:
methodology_snippet = r"""
Document embeddings were generated using the selected SentenceTransformer
model reported in the reproducible notebook outputs. As a robustness check,
the final embedding model was compared with a lighter SentenceTransformer
alternative using topic coherence, topic diversity, outlier ratio, and
qualitative inspection of representative abstracts. Silhouette score and
Davies--Bouldin index were retained as supplementary clustering
diagnostics for appendix reporting. The final configuration
was retained because it provided the most suitable balance between
interpretability and computational feasibility.

To support reproducibility without requiring a private API key, an additional
local LLM-assisted representation step can be incorporated using Llama 3.1
through Ollama. The LLM receives only the representative c-TF-IDF terms and
selected representative abstracts; it does not modify embeddings, clusters, or
document-topic assignments. Final topic labels must be manually validated
against the representative terms and abstracts before being reported in the
thesis.
"""

llama_footnote = r"""
For further details, see the
\href{https://maartengr.github.io/BERTopic/getting_started/representation/llm.html}
{BERTopic LLM representation documentation} and the
\href{https://docs.ollama.com/api/introduction}
{Ollama local API documentation}.
"""

(OUTPUT_DIR / "latex" / "methodology_snippet_llama31_local.tex").write_text(
    methodology_snippet + "\n" + llama_footnote,
    encoding="utf-8",
)
print(methodology_snippet)


## 14. Export Workbook, HTML Report, and ZIP


In [ ]:
assignments_cols = ["topic", "topic_probability", "title_clean", "abstract_clean", "abstract_word_count"]
if year_col and year_col in topic_df.columns:
    topic_df["publication_year"] = topic_df[year_col].values
    assignments_cols.insert(0, "publication_year")
if source_col and source_col in topic_df.columns:
    topic_df["source_title"] = topic_df[source_col].values
    assignments_cols.insert(1 if "publication_year" in assignments_cols else 0, "source_title")
if doi_col and doi_col in topic_df.columns:
    topic_df["doi"] = topic_df[doi_col].values
    assignments_cols.insert(0, "doi")

topic_df[assignments_cols].to_csv(OUTPUT_DIR / "tables" / "selected_document_topic_assignments.csv", index=False)

excel_path = OUTPUT_DIR / "tables" / "bertopic_reanalysis_results_workbook.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    quality_summary.to_excel(writer, sheet_name="abstract_quality", index=False)
    grid_results.to_excel(writer, sheet_name="grid_results", index=False)
    diagnostics.to_excel(writer, sheet_name="selected_diagnostics", index=False)
    topic_table.to_excel(writer, sheet_name="topic_inventory", index=False)
    labeling_evidence.to_excel(writer, sheet_name="labeling_evidence", index=False)
    topic_df[assignments_cols].to_excel(writer, sheet_name="assignments", index=False)

manifest = {
    "run_timestamp": datetime.now().isoformat(timespec="seconds"),
    "project_root": str(PROJECT_ROOT),
    "data_path": str(DATA_PATH),
    "thesaurus_path": str(THESAURUS_PATH),
    "seed": SEED,
    "preprocessing_version": PREPROCESSING_VERSION,
    "selected_run_id": SELECTED_RUN_ID,
    "selected_meta": selected_meta,
    "outputs": {
        "workbook": str(excel_path),
        "grid_results": str(GRID_RESULTS_PATH),
        "topic_inventory": str(OUTPUT_DIR / "tables" / "selected_topic_inventory_provisional.csv"),
        "labeling_evidence": str(OUTPUT_DIR / "labeling_evidence" / "representative_abstracts_for_labeling.csv"),
    },
}
(OUTPUT_DIR / "reports" / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

html = f"""
<html>
<head>
  <meta charset="utf-8">
  <title>BERTopic Thesis Reanalysis Report</title>
  <style>
    body {{ font-family: Georgia, serif; max-width: 1100px; margin: 30px auto; color: #222; }}
    h1, h2 {{ color: #2f4f4f; }}
    table {{ border-collapse: collapse; width: 100%; margin-bottom: 22px; font-size: 14px; }}
    th, td {{ border-bottom: 1px solid #ddd; padding: 7px; text-align: left; vertical-align: top; }}
    th {{ background: #d9d9d9; }}
    img {{ max-width: 100%; border: 1px solid #ccc; margin: 12px 0 24px 0; }}
    .note {{ color: #555; font-size: 14px; }}
  </style>
</head>
<body>
  <h1>BERTopic Thesis Reanalysis Report</h1>
  <p class="note">Generated automatically from the reproducible notebook.</p>
  <h2>Selected model diagnostics</h2>
  {diagnostics.to_html(index=False)}
  <h2>Embedding and hyperparameter grid</h2>
  {grid_results.to_html(index=False)}
  <h2>Topic inventory</h2>
  {topic_table.to_html(index=False)}
  <h2>Figures</h2>
  <img src="../figures/embedding_hyperparameter_grid_summary.png">
  <img src="../figures/selected_topic_prevalence.png">
  <img src="../figures/selected_representative_ctfidf_terms.png">
  <img src="../figures/selected_bertopic_umap_topic_map.png">
</body>
</html>
"""
html_path = OUTPUT_DIR / "html" / "bertopic_reanalysis_report.html"
html_path.write_text(html, encoding="utf-8")

zip_base = str(PROJECT_ROOT / "bertopic_reanalysis_outputs")
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print("Workbook:", excel_path)
print("HTML report:", html_path)
print("ZIP:", zip_path)
